# Tuning logistic regression and LightGBM

This notebook tunes the two main learned models using chronological training and validation partitions. Macro-F1 on validation data is used for selection. The test partition remains untouched until the final evaluation.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from boosted_preprocessing import (
    make_boosted_tree_preprocessor,
    prepare_boosted_tree_features,
)
from model_data import RAW_MODEL_COLUMNS, prepare_model_data
from model_evaluation import macro_f1_test
from models import make_multinomial_logistic_regression
from prep_job import (
    add_split_time,
    add_target,
    chronological_split,
    load_monthly_data,
    preprocess_partitions,
)

## Prepare both model-data paths

Each preprocessor is fitted on training rows only. Logistic regression uses sparse one-hot features; LightGBM receives the categorical features directly.

In [ ]:
linear_data = prepare_model_data(PROJECT_ROOT / "data")

raw_data = load_monthly_data(PROJECT_ROOT / "data", RAW_MODEL_COLUMNS)
labelled_data = add_target(raw_data)
split_data = chronological_split(add_split_time(labelled_data))
processed_data = preprocess_partitions(split_data)

train_features, y_train_boosted = prepare_boosted_tree_features(
    processed_data["train"]
)
validation_features, y_validation_boosted = prepare_boosted_tree_features(
    processed_data["validation"]
)
    
boosted_preprocessor = make_boosted_tree_preprocessor()
X_train_boosted = boosted_preprocessor.fit_transform(train_features)
X_validation_boosted = boosted_preprocessor.transform(validation_features)

target_names = sorted(y_train_boosted.unique())
target_to_id = {name: index for index, name in enumerate(target_names)}
y_train_boosted = y_train_boosted.map(target_to_id)
y_validation_boosted = y_validation_boosted.map(target_to_id)

pd.DataFrame({
    "partition": ["train", "validation"],
    "linear_rows": [len(linear_data.y_train), len(linear_data.y_validation)],
    "linear_features": [linear_data.X_train.shape[1], linear_data.X_validation.shape[1]],
    "boosted_features": [X_train_boosted.shape[1], X_validation_boosted.shape[1]],
})

## Tune logistic regression

The regularization strength `C` is selected by validation macro-F1. All candidates use the same class balancing and sparse preprocessing.

In [ ]:
logistic_candidates = {
    f"logistic_C_{C}": make_multinomial_logistic_regression(C=C, max_iter=100)
    for C in [0.003, 0.01, 0.03, 0.1]
}

logistic_results = []
for name, model in logistic_candidates.items():
    model.fit(linear_data.X_train, linear_data.y_train)
    logistic_results.append(
        macro_f1_test(
            model,
            linear_data.X_validation,
            linear_data.y_validation,
            model_name=name,
        )
    )

logistic_results = (
    pd.DataFrame(logistic_results)
    .sort_values(["macro_f1", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
logistic_results

## Tune LightGBM

These deliberately small candidates vary tree complexity and learning rate. LightGBM is configured for CPU because the installed wheel is not GPU-enabled. The test partition is not used for tuning.

In [6]:
categorical_columns = [
    column
    for column in X_train_boosted.columns
    if str(X_train_boosted[column].dtype) == "category"
]

lightgbm_candidates = {
    "lightgbm_200_trees": LGBMClassifier(
        objective="multiclass",
        num_class=len(target_names),
        n_estimators=200,
        num_leaves=31,
        learning_rate=0.1,
        class_weight="balanced",
        device_type="cpu",
        verbosity=-1,
        random_state=42,
    ),
    "lightgbm_400_trees": LGBMClassifier(
        objective="multiclass",
        num_class=len(target_names),
        n_estimators=400,
        num_leaves=63,
        learning_rate=0.1,
        class_weight="balanced",
        device_type="cpu",
        verbosity=-1,
        random_state=42,
    ),
    "lightgbm_600_trees_lower_rate": LGBMClassifier(
        objective="multiclass",
        num_class=len(target_names),
        n_estimators=600,
        num_leaves=31,
        learning_rate=0.05,
        class_weight="balanced",
        device_type="cpu",
        verbosity=-1,
        random_state=42,
    ),
}

lightgbm_results = []
for name, model in lightgbm_candidates.items():
    model.fit(
        X_train_boosted,
        y_train_boosted,
        categorical_feature=categorical_columns,
    )
    lightgbm_results.append(
        macro_f1_test(
            model,
            X_validation_boosted,
            y_validation_boosted,
            model_name=name,
        )
    )

lightgbm_results = (
    pd.DataFrame(lightgbm_results)
    .sort_values(["macro_f1", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
lightgbm_results

,model,n_test,macro_f1,accuracy
0,lightgbm_600_trees_lower_rate,2120356,0.326504,0.394425
1,lightgbm_200_trees,2120356,0.325796,0.394038
2,lightgbm_400_trees,2120356,0.325683,0.392670


## Compare tuned candidates

Select the best logistic-regression and LightGBM configurations using validation macro-F1. Only the selected configuration should be evaluated on the untouched test partition in the final evaluation notebook or final section.

In [ ]:
tuning_results = pd.concat(
    [logistic_results, lightgbm_results],
    ignore_index=True,
).sort_values(["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)
tuning_results

The selected hyperparameters and validation results should be recorded in the report. Logistic regression remains the linear reference, while LightGBM is the main nonlinear candidate.